In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Compact12 已保存结果：本地汇总与 ZIP 归档

先停止旧汇总单元，保留运行时及 /content，然后直接 Run all；无需 GPU。只读取原四片和 repair01 已保存的后评、质量与成本记录，不执行模型、codec、原始数组回放或缺口补算。首次将所需小文件并发复制到本地，统计完成后将完整 ZIP 保存到正式结果目录，旧结果保留。


In [ ]:
SOURCE_REF = 'main'
SOURCE_DIRECTORY = None
PREVIOUS_RUNTIME_STOPPED = True  # 已确认原四片 runtime 均 CLOSED
CONFIG = dict(
    parent='/content/drive/MyDrive/Video-WM_Result/baselines/public-native-frameindex-compact12-v1/attempt01',
    output='/content/drive/MyDrive/Video-WM_Result/baselines/public-native-frameindex-compact12-v1/attempt01/recovery/repair01',
    cache='/content/drive/MyDrive/Video-WM_Cache/baseline-compact12-v1',
    old='/content/drive/MyDrive/Video-WM_Result/baselines/public-native-and-frameindex-fpr001-v4/attempt01',
    old_cache='/content/drive/MyDrive/Video-WM_Cache/baseline-v4')
CONFIG['tables_only'] = True


运行阶段：STAGING（复制数/计划数）→ AGGREGATING（方法/分组）→ QUALITY_TABLES → WRITING_TABLES → PACKING → UPLOADING → SAVED。首次仍需读取 Drive 中已有记录；模型权重和媒体缓存不复制。未知/缺失结果与原分母保持。完成后的 ZIP 包含局部输入快照、全局表和结果说明；中断上传只留下 .partial，不冒充完成包。


In [ ]:
from pathlib import Path
import urllib.request, importlib.util
name = 'native_baseline_compact_recovery_v1_entry.py'
if SOURCE_DIRECTORY:
    path = Path(SOURCE_DIRECTORY)/'scripts'/name
else:
    path = Path('/content')/name
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/{SOURCE_REF}/scripts/{name}', path)
spec = importlib.util.spec_from_file_location('compact_repair_entry', path)
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
RUN = entry.execute(CONFIG, SOURCE_DIRECTORY, SOURCE_REF, PREVIOUS_RUNTIME_STOPPED)


新恢复包位于正式结果的 `recovery/repair01`，旧分片仍只读。`cpu_recovery.json` 保存逐槽恢复/剩余清单，`physical_recovery.json` 保存补算返回与失败。汇总所需记录并发复制到 `/content`，在本地执行原统计，再将输入记录和全局 A/B/C/D 表打包为 `aggregation_runs/<本轮目录>/global_results.zip` 保存到 Drive。进度显示复制数量、统计阶段、打包与上传；旧 `global` 目录保持原样。运行完成、测量完整和科学结论分别报告；保留原阈值、CAL32/TEST500、12 视图、DCT2＋CDC32 和旧 H/L。


In [ ]:
print('返回:', RUN)
print('汇总 ZIP 索引:', CONFIG['output']+'/aggregation_latest.json')
